In [1]:
!pip install torch_geometric
# Optional: install additional dependencies for better performance, aligning with PyTorch 2.3.0 and CUDA 12.1 wheels
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.0+cu121.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.3 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.3.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 124.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 133.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 123.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.6/949.6 kB 66.7 MB/s eta 0:00:00


In [2]:
!pip install streamlit biopython
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 123.3 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [3]:
%%writefile app.py
import streamlit as st
import torch
import numpy as np
import os
import re
import io
from Bio import SeqIO
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
import matplotlib.pyplot as plt
import pandas as pd

# --- Model Definition ---
class ProphageSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(ProphageSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

def create_graph_from_embeddings(embeddings_numpy):
    x = torch.from_numpy(embeddings_numpy).float()
    num_nodes = x.size(0)
    edge_index = torch.stack([
        torch.cat([torch.arange(0, num_nodes - 1), torch.arange(1, num_nodes)]),
        torch.cat([torch.arange(1, num_nodes), torch.arange(0, num_nodes - 1)])
    ], dim=0)
    return Data(x=x, edge_index=edge_index)

def extract_regions(probs, threshold, min_length, gap_tolerance):
    regions = []; current_region = []; gap_count = 0
    for i, p in enumerate(probs):
        if p >= threshold:
            current_region.append(i)
            gap_count = 0
        elif current_region:
            gap_count += 1
            if gap_count > gap_tolerance:
                if len(current_region) >= min_length: regions.append((current_region[0], current_region[-1]))
                current_region = []; gap_count = 0
            else: current_region.append(i)
    if len(current_region) >= min_length: regions.append((current_region[0], current_region[-1]))
    return regions

def get_genomic_coords(records, predicted_regions):
    final_coordinates = []
    for start_idx, end_idx in predicted_regions:
        if 0 <= start_idx < len(records) and 0 <= end_idx < len(records):
            desc_s, desc_e = records[start_idx].description, records[end_idx].description
            coords_start = re.findall(r'# (\d+) # (\d+) #', desc_s)
            coords_end = re.findall(r'# (\d+) # (\d+) #', desc_e)
            if coords_start and coords_end:
                final_coordinates.append({'Protein Range': f'{start_idx}-{end_idx}', 'Genomic Start': coords_start[0][0], 'Genomic End': coords_end[0][1]})
    return final_coordinates

st.set_page_config(page_title="ProSAGE")
st.title("ProSAGE: Prophage Prediction")
threshold = st.sidebar.slider("Viral Threshold", 0.0, 1.0, 0.6)
min_proteins = st.sidebar.number_input("Min Proteins", 5, 100, 10)
gap_tol = st.sidebar.number_input("Gap Tolerance", 0, 10, 2)
up_emb = st.file_uploader("Upload Genome Embeddings (.npy)", type=['npy'])
up_faa = st.file_uploader("Upload Prodigal Output (.faa)", type=['faa', 'fasta'])

if up_emb and up_faa:
    emb = np.load(up_emb)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    emb = emb / (norms + 1e-12)

    # Fix for StreamModeError: wrap bytes in a text stream
    stringio = io.StringIO(up_faa.getvalue().decode("utf-8"))
    records = list(SeqIO.parse(stringio, "fasta"))

    if st.button("Run Prediction"):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = ProphageSAGE(1280, 256, 2).to(device)
        if os.path.exists('prophage_sage_model.pt'):
            checkpoint = torch.load('prophage_sage_model.pt', map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            model.eval()
            data = create_graph_from_embeddings(emb).to(device)
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            regions = extract_regions(probs, threshold, min_proteins, gap_tol)
            coords = get_genomic_coords(records, regions)
            st.subheader("Results")
            if coords:
                st.dataframe(pd.DataFrame(coords))
            else:
                st.write("No prophages found.")
        else:
            st.error("Missing prophage_sage_model.pt")

Writing app.py


In [4]:
import subprocess
import os
import time

# 1. Download and install bore binary
if not os.path.exists('bore'):
    print("Installing bore...")
    !wget -q https://github.com/ekzhang/bore/releases/download/v0.5.1/bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz
    !tar -xzf bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz
    !chmod +x bore

# 2. Start Streamlit in the background
print("Starting Streamlit...")
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0'])
time.sleep(5)

# 3. Run bore to expose the port
print("\nConnecting to bore.pub...")
print("Your app will be available at: bore.pub:[REMOTE_PORT]")
!./bore local 8501 --to bore.pub

Installing bore...
Starting Streamlit...

Connecting to bore.pub...
Your app will be available at: bore.pub:[REMOTE_PORT]
2026-07-06T10:16:45.437081Z  INFO bore_cli::client: connected to server remote_port=5662
2026-07-06T10:16:45.437116Z  INFO bore_cli::client: listening at bore.pub:5662
2026-07-06T10:17:00.797614Z  INFO proxy{id=3d60b4a1-4a98-4107-bf98-20c588049349}: bore_cli::client: new connection
2026-07-06T10:17:01.090220Z  INFO proxy{id=5896e43c-1896-4f4c-889b-4ac85f5ac915}: bore_cli::client: new connection
2026-07-06T10:17:01.560309Z  INFO proxy{id=55d6f79b-3b38-47db-a1a1-139474c84f0d}: bore_cli::client: new connection
2026-07-06T10:17:01.630864Z  INFO proxy{id=0c986657-edc1-4def-89f7-3b4fc84c05e7}: bore_cli::client: new connection
2026-07-06T10:17:06.312077Z  INFO proxy{id=5896e43c-1896-4f4c-889b-4ac85f5ac915}: bore_cli::client: connection exited
2026-07-06T10:17:06.492071Z  INFO proxy{id=3d60b4a1-4a98-4107-bf98-20c588049349}: bore_cli::client: connection exited
2026-07-06T10: